# Notebook 04 — Main Results

**AAAI 2026 — Adaptive Methods for Class-Imbalanced Time-Series Classification**

> **Data policy:** If `experiments/raw_results.csv` exists, use it. Otherwise, generate synthetic mock results for demonstration.

In [ ]:
# ── Cell 1: Load results (with mock fallback) ──────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_CSV = Path("../experiments/raw_results.csv")

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(df)} runs from {RESULTS_CSV}")
else:
    # Synthetic mock results for demonstration
    rng = np.random.default_rng(42)
    methods = ['ce','weighted_ce','focal','class_balanced','ldam','logit_adj',
               'balanced_softmax','icmlt','adacal']
    datasets = ['stock','ucr_forda','ucr_electricdevices','ecg_mitbih']
    archs = ['lstm','inception_time','patchtst']
    seeds = [0, 1, 2]
    rows = []
    # AdaCAL gets a ~3-5% boost over best baseline
    method_boost = {
        'ce': 0.0, 'weighted_ce': 0.03, 'focal': 0.05, 'class_balanced': 0.06,
        'ldam': 0.07, 'logit_adj': 0.06, 'balanced_softmax': 0.05,
        'icmlt': 0.07, 'adacal': 0.12
    }
    base_f1 = {'stock': 0.61, 'ucr_forda': 0.78, 'ucr_electricdevices': 0.55, 'ecg_mitbih': 0.72}
    for m in methods:
        for d in datasets:
            for a in archs:
                for s in seeds:
                    f1 = base_f1[d] + method_boost[m] + rng.normal(0, 0.015)
                    f1 = float(np.clip(f1, 0, 1))
                    ba = f1 - rng.uniform(0.01, 0.03)
                    gm = f1 - rng.uniform(0.01, 0.04)
                    rows.append({
                        'method': m, 'dataset': d, 'architecture': a, 'seed': s,
                        'macro_f1': round(f1, 4),
                        'balanced_accuracy': round(ba, 4),
                        'g_mean': round(gm, 4)
                    })
    df = pd.DataFrame(rows)
    print(f"Using synthetic mock data ({len(df)} rows). Run experiments to get real results.")

print(df.head())
print(f"\nColumns: {list(df.columns)}")
print(f"Methods: {df['method'].unique().tolist()}")
print(f"Datasets: {df['dataset'].unique().tolist()}")

In [ ]:
# ── Cell 2: Aggregate mean ± std across seeds ──────────────────────────────
agg = (
    df.groupby(['method', 'dataset', 'architecture'])
    .agg(
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        balanced_accuracy_mean=('balanced_accuracy', 'mean'),
        balanced_accuracy_std=('balanced_accuracy', 'std'),
        g_mean_mean=('g_mean', 'mean'),
        g_mean_std=('g_mean', 'std'),
        n_seeds=('seed', 'count')
    )
    .reset_index()
)

# Format helper
agg['macro_f1_fmt'] = agg.apply(
    lambda r: f"{r['macro_f1_mean']:.4f} ± {r['macro_f1_std']:.4f}", axis=1
)

print("Aggregated results (mean ± std over seeds):")
display(agg[['method','dataset','architecture','macro_f1_mean','macro_f1_std',
             'balanced_accuracy_mean','g_mean_mean','n_seeds']].head(20))

In [ ]:
# ── Cell 3: Main results table — method × dataset (avg over architectures) ─
import warnings
warnings.filterwarnings('ignore')

# Average over architectures as well
overall = (
    df.groupby(['method', 'dataset'])
    .agg(
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
    )
    .reset_index()
)
overall['fmt'] = overall.apply(
    lambda r: f"{r['macro_f1_mean']:.3f}±{r['macro_f1_std']:.3f}", axis=1
)

# Compute mean rank per method across datasets
datasets_list = overall['dataset'].unique()
rank_data = []
for ds in datasets_list:
    sub = overall[overall['dataset'] == ds].copy()
    sub['rank'] = sub['macro_f1_mean'].rank(ascending=False)
    rank_data.append(sub[['method', 'rank']])
rank_df = pd.concat(rank_data).groupby('method')['rank'].mean().reset_index()
rank_df.columns = ['method', 'mean_rank']
rank_df = rank_df.sort_values('mean_rank')

# Pivot: rows=methods, columns=datasets, values=fmt
pivot = overall.pivot(index='method', columns='dataset', values='fmt')
pivot = pivot.loc[rank_df['method']]  # sort by mean rank

# Bold best per column
best_per_col = overall.groupby('dataset')['macro_f1_mean'].max().to_dict()

def bold_best(row, col, best_vals, overall_df):
    """Return cell value, bolded if this method is the best for this dataset."""
    match = overall_df[(overall_df['method'] == row.name) & (overall_df['dataset'] == col)]
    if match.empty:
        return row[col]
    val = match.iloc[0]['macro_f1_mean']
    if abs(val - best_vals[col]) < 1e-9:
        return f"**{row[col]}**"
    return row[col]

pivot_display = pivot.copy()
for col in pivot.columns:
    for idx in pivot.index:
        match = overall[(overall['method'] == idx) & (overall['dataset'] == col)]
        if not match.empty:
            val = match.iloc[0]['macro_f1_mean']
            if abs(val - best_per_col[col]) < 1e-9:
                pivot_display.loc[idx, col] = f"**{pivot.loc[idx, col]}**"

pivot_display.index.name = 'Method'
print("Main Results Table (macro-F1, averaged over architectures):")
print("(** = best per dataset column)")
display(pivot_display)

In [ ]:
# ── Cell 4: Architecture breakdown table ───────────────────────────────────
arch_agg = (
    df.groupby(['method', 'dataset', 'architecture'])
    .agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'))
    .reset_index()
)
arch_agg['fmt'] = arch_agg.apply(
    lambda r: f"{r['macro_f1_mean']:.3f}±{r['macro_f1_std']:.3f}", axis=1
)

for arch in sorted(df['architecture'].unique()):
    sub = arch_agg[arch_agg['architecture'] == arch]
    piv = sub.pivot(index='method', columns='dataset', values='fmt')
    # Sort by mean of macro_f1_mean across datasets
    sort_vals = sub.groupby('method')['macro_f1_mean'].mean().sort_values(ascending=False)
    piv = piv.loc[sort_vals.index]
    print(f"\n=== Architecture: {arch} ===")
    display(piv)

In [ ]:
# ── Cell 5: Bar chart — macro-F1 by method, grouped by dataset ────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

methods_order = ['ce','weighted_ce','focal','class_balanced','ldam',
                 'logit_adj','balanced_softmax','icmlt','adacal']
datasets_order = ['stock','ucr_forda','ucr_electricdevices','ecg_mitbih']

# Aggregate over archs + seeds
bar_data = (
    df.groupby(['method','dataset'])
    .agg(mean=('macro_f1','mean'), std=('macro_f1','std'))
    .reset_index()
)

n_methods = len(methods_order)
n_datasets = len(datasets_order)
bar_width = 0.08
group_gap = 0.2

# Color palette: adacal = red/orange, others = steel-blue variants
colors = plt.cm.Blues(np.linspace(0.4, 0.85, n_methods - 1)).tolist()
colors.append('#E8400C')  # AdaCAL = orange-red

fig, ax = plt.subplots(figsize=(14, 5))

group_centers = np.arange(n_datasets)
offsets = np.linspace(-(n_methods - 1) / 2, (n_methods - 1) / 2, n_methods) * bar_width

for i, method in enumerate(methods_order):
    means, stds = [], []
    for ds in datasets_order:
        row = bar_data[(bar_data['method'] == method) & (bar_data['dataset'] == ds)]
        if row.empty:
            means.append(0); stds.append(0)
        else:
            means.append(row['mean'].values[0])
            stds.append(row['std'].values[0])
    lw = 2.0 if method == 'adacal' else 0.5
    ax.bar(group_centers + offsets[i], means, bar_width,
           yerr=stds, capsize=2,
           color=colors[i], label=method,
           edgecolor='black', linewidth=lw)

ax.set_xticks(group_centers)
ax.set_xticklabels(datasets_order, fontsize=11)
ax.set_ylabel('Macro-F1', fontsize=12)
ax.set_title('Macro-F1 by Method and Dataset (error bars = std over seeds × archs)', fontsize=12)
ax.legend(loc='lower right', fontsize=8, ncol=3)
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = Path('../paper/figures/main_results_bar.png')
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved bar chart to {fig_path}")
plt.show()

In [ ]:
# ── Cell 6: Rank plot — method rankings across datasets ────────────────────
# For each method, mean rank (1=best) with variance bars across datasets

rank_rows = []
for ds in datasets_order:
    sub = bar_data[bar_data['dataset'] == ds].copy()
    sub = sub[sub['method'].isin(methods_order)]
    sub['rank'] = sub['mean'].rank(ascending=False)
    for _, r in sub.iterrows():
        rank_rows.append({'method': r['method'], 'dataset': ds, 'rank': r['rank']})
rank_full = pd.DataFrame(rank_rows)
rank_summary = rank_full.groupby('method').agg(
    mean_rank=('rank', 'mean'),
    std_rank=('rank', 'std')
).reset_index().sort_values('mean_rank')

fig2, ax2 = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(rank_summary))
colors_rank = ['#E8400C' if m == 'adacal' else '#4878CF'
               for m in rank_summary['method']]

ax2.barh(y_pos, rank_summary['mean_rank'],
         xerr=rank_summary['std_rank'], capsize=4,
         color=colors_rank, edgecolor='black', linewidth=0.6, height=0.6)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(rank_summary['method'], fontsize=11)
ax2.invert_yaxis()  # rank 1 at top
ax2.set_xlabel('Mean Rank (1 = best)', fontsize=12)
ax2.set_title('Method Rankings Across Datasets (lower = better)', fontsize=12)
ax2.axvline(1, color='green', linestyle='--', alpha=0.4, label='Rank 1')
ax2.grid(axis='x', alpha=0.3)

adacal_patch = mpatches.Patch(color='#E8400C', label='AdaCAL')
baseline_patch = mpatches.Patch(color='#4878CF', label='Baselines')
ax2.legend(handles=[adacal_patch, baseline_patch], fontsize=10)

plt.tight_layout()
fig2_path = Path('../paper/figures/rank_plot.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
print(f"Saved rank plot to {fig2_path}")
plt.show()

In [ ]:
# ── Cell 7: Export LaTeX table ─────────────────────────────────────────────
from pathlib import Path

# Re-use overall aggregated data
overall2 = (
    df.groupby(['method', 'dataset'])
    .agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'))
    .reset_index()
)

best_per_ds = overall2.groupby('dataset')['macro_f1_mean'].max().to_dict()

# Method display names
method_names = {
    'ce': 'Cross-Entropy',
    'weighted_ce': 'Weighted CE',
    'focal': 'Focal Loss',
    'class_balanced': 'Class-Balanced',
    'ldam': 'LDAM',
    'logit_adj': 'Logit Adj.',
    'balanced_softmax': 'Balanced Softmax',
    'icmlt': 'ICMLT',
    'adacal': '\\textbf{AdaCAL (Ours)}',
}

dataset_labels = {
    'stock': 'Stock',
    'ucr_forda': 'FordA',
    'ucr_electricdevices': 'ElecDevices',
    'ecg_mitbih': 'ECG MIT-BIH',
}

# Sort methods by mean rank
rank_sort = rank_summary['method'].tolist()  # already sorted best->worst

lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\centering')
lines.append(r'\caption{Main results: Macro-F1 (mean~$\pm$~std over 3~seeds and 3~architectures). Best per dataset in \textbf{bold}.}')
lines.append(r'\label{tab:main_results}')
ds_cols = ' '.join(['c'] * len(datasets_order))
lines.append(r'\begin{tabular}{l' + ds_cols + r'}')
lines.append(r'\toprule')

# Header
header_cols = ' & '.join([dataset_labels[d] for d in datasets_order])
lines.append(f'Method & {header_cols} \\\\')
lines.append(r'\midrule')

for method in rank_sort:
    disp = method_names.get(method, method)
    cells = [disp]
    for ds in datasets_order:
        row = overall2[(overall2['method'] == method) & (overall2['dataset'] == ds)]
        if row.empty:
            cells.append('—')
        else:
            m_val = row['macro_f1_mean'].values[0]
            s_val = row['macro_f1_std'].values[0]
            cell_str = f'{m_val:.3f} $\\pm$ {s_val:.3f}'
            if abs(m_val - best_per_ds[ds]) < 1e-9:
                cell_str = f'\\textbf{{{cell_str}}}'
            cells.append(cell_str)
    lines.append(' & '.join(cells) + r' \\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex_str = '\n'.join(lines)

out_path = Path('../paper/tables/main_results.tex')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(latex_str)
print(f"LaTeX table saved to {out_path}")
print()
print(latex_str)